# Survival, workout LGD, and revolving EAD

Construct lifetime default curves, discounted recoveries, raw LGD, and credit conversion factors with visible boundary adjustments.

All data are generated locally unless this notebook explicitly calls a reviewed adapter. Results are educational and require independent validation before any real use.

In [ ]:
import numpy as np
import pandas as pd

from creditriskbook.risk_components import calculate_workout_lgd, construct_ccf, ead_from_ccf
from creditriskbook.survival import cumulative_pd_from_hazard, kaplan_meier

durations = np.array([3, 5, 5, 7, 9, 12, 12, 18, 24, 24], dtype=float)
events = np.array([1, 1, 0, 1, 0, 1, 0, 1, 0, 0])
curve = kaplan_meier(durations, events)
hazard_pd = cumulative_pd_from_hazard(np.array([0.02, 0.03, 0.04, 0.05]))
print(curve)
print("Cumulative PD from hazard:", hazard_pd)

In [ ]:
ledger = pd.DataFrame({
    "account_id": ["A", "A", "B", "B"],
    "default_date": ["2024-01-01"] * 4,
    "cashflow_date": ["2024-04-01", "2025-01-01", "2024-02-01", "2024-08-01"],
    "recovery": [2_000, 3_000, 7_000, 5_000],
    "direct_cost": [100, 150, 300, 200],
    "ead_at_default": [10_000] * 4,
    "effective_interest_rate": [0.08] * 4,
})
lgd = calculate_workout_lgd(ledger)
print(lgd)
assert {"lgd_raw", "lgd_model", "boundary_adjustment"}.issubset(lgd)

In [ ]:
facilities = pd.DataFrame({
    "facility_id": ["F1", "F2", "F3"],
    "drawn_reference": [4_000, 8_000, 2_000],
    "limit_reference": [10_000, 10_000, 5_000],
    "ead_at_default": [7_000, 9_500, 4_400],
})
ccf = construct_ccf(facilities)
ccf["ead_rebuilt"] = ead_from_ccf(ccf["drawn_reference"], ccf["undrawn_reference"], ccf["ccf_model"])
print(ccf)